# 📘 Notebook 08: Machine Learning: Concepts & Workflow

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module C: Machine Learning · Notebook 1 of 4 in this module · (8 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Define machine learning and distinguish it from traditional programming
- Tell apart **supervised**, **unsupervised**, and (briefly) **reinforcement** learning
- Explain the standard ML **workflow** end to end
- Understand the critical **train/test split** and *why* we hold data back
- Reason about **overfitting**, **underfitting**, and the **bias-variance trade-off**
- Know the common **evaluation metrics** and when each is appropriate

> **Prerequisites:** Modules A and B.
>
> This notebook is mostly **conceptual**, the ideas here are the mental scaffolding for everything that follows. Take it slowly; a clear grasp of these fundamentals matters more than any single algorithm.

> ℹ️ **Setup note.** Later cells use scikit-learn. If needed, run once:
```python
import piplite
await piplite.install(['scikit-learn', 'numpy', 'matplotlib'])
```

## 1. What is machine learning?

### Traditional programming vs machine learning
In **traditional programming**, a human writes explicit rules: *input + rules → output*. To detect spam, you might write ‘if the email contains the word *lottery*, mark it as spam’. This breaks down for complex problems, you cannot hand-write rules to recognise a cat in a photo.

**Machine learning inverts this.** Instead of writing the rules, we show the computer many **examples** of inputs together with their correct outputs, and the algorithm *learns the rules itself*: *input + output → rules (a model)*.

### A definition
> A computer program is said to **learn** from experience *E* with respect to some task *T* and performance measure *P*, if its performance at *T*, as measured by *P*, improves with experience *E*., Tom Mitchell

In plain terms: the model gets better at the task as it sees more data.

## 2. The three broad types of machine learning

### Supervised learning
The training data is **labelled**, each example comes with the correct answer. The model learns to map inputs to outputs. Two sub-types:
- **Regression**, predict a *continuous number* (house price, temperature).
- **Classification**, predict a *category* (spam / not spam, disease / no disease).

This is the most common type and the focus of Notebooks 09-10.

### Unsupervised learning
The data is **unlabelled**, there are no correct answers given. The model finds structure on its own, for example grouping similar items together (**clustering**) or compressing data (**dimensionality reduction**). Covered in Notebook 11.

### Reinforcement learning
An **agent** learns by interacting with an environment, receiving rewards or penalties for its actions (think of training a game-playing AI). Important, but beyond this course's scope.

### Vocabulary you will use constantly
| Term | Meaning |
|------|---------|
| **Feature** | An input variable (a column). Often written $X$. |
| **Label / target** | The output we want to predict. Often written $y$. |
| **Sample / instance** | One data point (a row). |
| **Model** | The learned mapping from features to target. |
| **Training** | The process of fitting the model to data. |

## 3. The machine-learning workflow

Almost every supervised ML project follows the same sequence of steps. Internalising this pipeline gives you a reliable structure for any problem:

1. **Define the problem**, what are we predicting, and is it regression or classification?
2. **Collect and explore data**, load it, visualise it, understand it (Modules B).
3. **Prepare the data**, clean missing values, encode categories, **scale** features.
4. **Split** the data, into a training set and a test set.
5. **Choose and train a model**, fit it on the training set.
6. **Evaluate**, measure performance on the *test* set with appropriate metrics.
7. **Tune and iterate**, adjust and repeat.
8. **Deploy**, put the model to use.

We will practise steps 3-7 repeatedly from Notebook 09 onward.

## 4. The train/test split: the most important habit in ML

### The core idea
If you let a student see the exam answers while studying, a perfect exam score tells you nothing about whether they learned. The same is true for models. We must evaluate a model on data it has **never seen during training**.

So we **split** our data:
- the **training set** (typically 70-80%) is used to fit the model;
- the **test set** (the remaining 20-30%) is locked away and used only at the end to estimate real-world performance.

### Why this matters so much
A model that has effectively memorised its training data can score perfectly on it while failing completely on new data. The test set is our honest estimate of **generalisation**, performance on the unseen.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Toy data: 10 samples, one feature X and target y
X = np.arange(10).reshape(-1, 1)   # shape (10, 1)
y = np.array([0,0,0,0,0,1,1,1,1,1])

# Hold back 30% for testing. random_state makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

> ⚠️ **Common pitfalls**
>
> - **Data leakage**: any information from the test set sneaking into training (e.g. scaling using statistics computed over the whole dataset) gives falsely optimistic results. Always split *first*, then preprocess using only the training set.
> - Evaluating on the training set and reporting that as your accuracy. It is almost always too optimistic.

## 5. Overfitting, underfitting, and the bias-variance trade-off

### The two ways a model can fail
- **Underfitting (high bias):** the model is too simple to capture the pattern. It does poorly on *both* training and test data. Like trying to fit a straight line to a clearly curved trend.
- **Overfitting (high variance):** the model is too complex and memorises the training data, including its random noise. It does *great* on training data but *poorly* on test data, it failed to generalise.

### The trade-off
We want the sweet spot in between: complex enough to capture the real pattern, simple enough to ignore the noise. This tension between bias and variance is one of the central themes of machine learning. Let us make it **visual**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
# True relationship is a gentle curve; we add noise to simulate real data
X = np.linspace(0, 1, 20)
y_true = np.sin(2 * np.pi * X)
y = y_true + np.random.normal(0, 0.2, size=X.shape)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, degree, title in zip(axes, [1, 4, 15],
                             ["Underfit (degree 1)", "Good fit (degree 4)", "Overfit (degree 15)"]):
    coeffs = np.polyfit(X, y, degree)         # fit a polynomial of this degree
    xs = np.linspace(0, 1, 200)
    ys = np.polyval(coeffs, xs)
    ax.scatter(X, y, color="black", s=20, label="data")
    ax.plot(xs, ys, color="red", label="model")
    ax.plot(xs, np.sin(2*np.pi*xs), "g--", alpha=0.6, label="true")
    ax.set_title(title)
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Study the three panels:
- **Left (degree 1):** a straight line cannot follow the curve, *underfitting*.
- **Middle (degree 4):** the model tracks the true green curve closely without chasing every point, *the goal*.
- **Right (degree 15):** the red line wiggles wildly to pass through individual noisy points, *overfitting*. It would predict new points badly.

## 6. Evaluation metrics

How we measure ‘good’ depends on the task.

### For regression (predicting a number)
- **Mean Absolute Error (MAE):** average absolute difference, $\frac{1}{n}\sum |y_i - \hat{y}_i|$. Easy to interpret.
- **Mean Squared Error (MSE):** $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$. Punishes large errors more.
- **$R^2$ (coefficient of determination):** the fraction of variance explained; 1.0 is perfect, 0 means no better than predicting the mean.

### For classification (predicting a category)
- **Accuracy:** fraction of correct predictions. Simple, but *misleading on imbalanced data*.
- **Precision:** of the items predicted positive, how many truly are. ($\frac{TP}{TP+FP}$)
- **Recall:** of the truly positive items, how many we caught. ($\frac{TP}{TP+FN}$)
- **F1-score:** the harmonic mean of precision and recall, a single balanced number.

> 🧠 **Why accuracy can lie.** If 99% of emails are *not* spam, a lazy model that labels *everything* ‘not spam’ scores 99% accuracy while catching zero spam. Precision, recall, and F1 expose this failure. Choosing the right metric is a judgement about what kind of mistake matters most in your problem.

---
## ✏️ Exercises

*These are mostly conceptual, write your answers in a text cell or as comments.*

### Exercise 1
For each task, state whether it is **regression**, **classification**, or **unsupervised**:
1. Predicting tomorrow's temperature in degrees.
2. Deciding whether a transaction is fraudulent.
3. Grouping customers into segments without predefined labels.
4. Predicting a house's sale price.

In [ ]:
# Your answers here (as comments):


<details><summary>💡 Show solution</summary>

```python
# 1. Regression (continuous number)
# 2. Classification (two categories: fraud / not fraud)
# 3. Unsupervised (clustering, no labels given)
# 4. Regression (continuous price)
```
</details>

### Exercise 2
A model scores 99% accuracy on a training set but 62% on the test set. Which problem is this, and what does it tell you?

In [ ]:
# Your answer here:


<details><summary>💡 Show solution</summary>

```python
# This is OVERFITTING (high variance).
# The large gap between training (99%) and test (62%) performance shows the
# model memorised the training data, including noise, and fails to generalise
# to unseen data. Remedies: simpler model, more data, or regularisation.
```

*The signature of overfitting is precisely this gap: excellent on training, poor on test.*
</details>

### Exercise 3
A medical screening test for a rare disease (1% of patients have it) reports 99% accuracy. Why is this not necessarily a good test, and which metric would reveal the truth?

In [ ]:
# Your answer here:


<details><summary>💡 Show solution</summary>

```python
# A model that predicts 'no disease' for EVERYONE is automatically 99% accurate,
# because 99% of patients are healthy -- yet it catches ZERO sick patients.
# RECALL (how many true cases we detect) would expose this: it would be 0%.
# For rare-but-important events, recall (and F1) matter far more than accuracy.
```
</details>

## 🔑 Key takeaways

- ML learns rules *from examples*, inverting traditional rule-writing.
- **Supervised** (labelled: regression & classification), **unsupervised** (unlabelled), reinforcement (reward-driven).
- The ML **workflow** is a repeatable pipeline from problem definition to deployment.
- Always **split** data and evaluate on an unseen **test set**, this estimates generalisation.
- **Underfitting** = too simple (high bias); **overfitting** = too complex, memorises noise (high variance).
- Choose metrics by task; **accuracy can mislead** on imbalanced data, precision, recall, and F1 tell the fuller story.

---
**Next:** *Notebook 09: Supervised Learning I: Regression*, where we build our first predictive models and even implement gradient descent by hand.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*